$$\newcommand{\ket}[1]{\left|{#1}\right\rangle}$$
$$\newcommand{\bra}[1]{\left\langle{#1}\right|}$$

# From a Molecule to a Qubit Hamiltonian

In the previous notebook we ran a VQE on this operator:

```python
hamiltonian = SparsePauliOp.from_list(
    [("YZ", 0.3980), ("ZI", -0.3980), ("ZZ", -0.0113), ("XX", 0.1810)]
)
```

and we found its lowest eigenvalue. But we never said where those four numbers came
from, and we never checked whether the number we computed was an *energy* of anything.
It is not: it is the smallest eigenvalue of a $4\times4$ matrix that was handed to you.

In this notebook we build the hydrogen Hamiltonian from scratch. The route is:

$$
\underbrace{\hat{H}(\mathbf{r})}_{\text{electrons in space}}
\;\longrightarrow\;
\underbrace{\hat{H} = \sum h_{pq}\,\hat{a}^\dagger_p \hat{a}_q + \tfrac12\sum h_{pqrs}\,\hat{a}^\dagger_p\hat{a}^\dagger_q\hat{a}_r\hat{a}_s}_{\text{second quantisation}}
\;\longrightarrow\;
\underbrace{\hat{H} = \sum_j c_j\, \hat{P}_j}_{\text{qubits}}
$$

By the end you will have derived a two-qubit Hamiltonian whose ground-state energy is
$-1.137\,306\ \mathrm{Ha}$, the exact non-relativistic energy of $\mathrm{H}_2$ in the
STO-3G basis, and you will have computed the full dissociation curve of the molecule
on a quantum simulator.

## What you need

Only `numpy`, `qiskit`, and the file `data/h2_sto3g.npz` that ships with this
repository. That file contains the molecular integrals, precomputed with
[PySCF](https://pyscf.org/) so that nobody has to install a quantum chemistry package
during the tutorial. The script that generated it is in `data/generate_integrals.py`
if you want to reproduce it or run a different molecule.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_algorithms.optimizers import COBYLA

backend = AerSimulator()
rng = np.random.default_rng(42)

---
# 1. The electronic structure problem

Within the Born--Oppenheimer approximation we freeze the nuclei and solve for the
electrons only. For a molecule with $N$ electrons and nuclei of charge $Z_I$ at
positions $\mathbf{R}_I$, the electronic Hamiltonian in atomic units is

\begin{equation}
    \hat{H} = -\sum_{i=1}^{N}\frac{\nabla_i^2}{2}
              - \sum_{i=1}^{N}\sum_{I}\frac{Z_I}{|\mathbf{r}_i - \mathbf{R}_I|}
              + \sum_{i<j}\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}
              + \underbrace{\sum_{I<J}\frac{Z_I Z_J}{|\mathbf{R}_I - \mathbf{R}_J|}}_{E_{\rm nuc},\ \text{a constant}}.
\end{equation}

This is a differential operator on $3N$ continuous coordinates, which is not something
a qubit register can represent. The standard fix is to pick a finite basis of
single-particle spatial orbitals $\{\phi_p(\mathbf{r})\}_{p=1}^{M}$ and to work in the
Fock space they generate.

We will use **STO-3G**, the smallest basis in common use: one $1s$-like function per
hydrogen atom, so $M = 2$ spatial orbitals and therefore $2M = 4$ **spin orbitals**.
This is far too small for chemical accuracy on any real problem, but it is exactly
solvable classically, which is precisely what we want while learning.

Projecting the Hamiltonian onto that basis leaves us with two tensors of numbers:

\begin{equation}
    h_{pq} = \int \mathrm{d}\mathbf{r}\; \phi_p^*(\mathbf{r})
    \left(-\frac{\nabla^2}{2} - \sum_I \frac{Z_I}{|\mathbf{r} - \mathbf{R}_I|}\right)\phi_q(\mathbf{r}),
\end{equation}

\begin{equation}
    (pq|rs) = \int\! \mathrm{d}\mathbf{r}_1 \mathrm{d}\mathbf{r}_2\;
    \frac{\phi_p^*(\mathbf{r}_1)\phi_q(\mathbf{r}_1)\,\phi_r^*(\mathbf{r}_2)\phi_s(\mathbf{r}_2)}{|\mathbf{r}_1 - \mathbf{r}_2|}.
\end{equation}

$h_{pq}$ is the one-electron (kinetic + nuclear attraction) integral and $(pq|rs)$ is
the two-electron repulsion integral in *chemists' notation*. Everything about the
molecule that a quantum computer will ever see is contained in these two arrays plus
the constant $E_{\rm nuc}$.

Let us load them.

In [ ]:
data = np.load("data/h2_sto3g.npz")

print("arrays in the file:")
for key in data.files:
    print(f"  {key:15s} {data[key].shape}")

bond_lengths = data["bond_lengths"]
print(f"\ngeometries: {len(bond_lengths)} bond lengths from "
      f"{bond_lengths[0]} to {bond_lengths[-1]} Angstrom")

In [ ]:
# Pick the equilibrium geometry to start with.
i_eq = int(np.argmin(np.abs(bond_lengths - 0.735)))
R = bond_lengths[i_eq]

h1    = data["h1"][i_eq]      # (M, M)      one-electron integrals h_pq
eri   = data["eri"][i_eq]     # (M,M,M,M)   two-electron integrals (pq|rs)
e_nuc = data["e_nuc"][i_eq]   # scalar      nuclear repulsion
e_hf  = data["e_hf"][i_eq]    # Hartree-Fock reference energy
e_fci = data["e_fci"][i_eq]   # exact (full configuration interaction) energy

print(f"R = {R} A\n")
print("h_pq =\n", np.round(h1, 5))
print(f"\nE_nuc = {e_nuc:.6f} Ha")
print(f"E_HF  = {e_hf:.6f} Ha   <- mean-field answer")
print(f"E_FCI = {e_fci:.6f} Ha   <- exact answer in this basis (our target)")
print(f"\ncorrelation energy = {1000*(e_fci - e_hf):.2f} mHa")

The orbitals here are the **Hartree--Fock molecular orbitals**: $\phi_0$ is the bonding
$\sigma_g$ combination and $\phi_1$ the antibonding $\sigma_u^*$. In this basis the
mean-field ground state is simply "both electrons in $\phi_0$", and the difference
between that and the true answer is the *correlation energy*. It is small here
(about 20 mHa) but it is the entire reason anyone wants a quantum computer for
chemistry: it is the part that mean-field theory cannot get.

Note the scale. Chemical accuracy is conventionally $1.6\ \mathrm{mHa}$
($1\ \mathrm{kcal/mol}$). Keep that number in mind when we start adding shot noise.

---
# 2. Second quantisation

Rather than tracking antisymmetrised wavefunctions of $N$ electrons, we track
*occupations* of spin orbitals. A spin orbital is a spatial orbital times a spin
function, $\chi_{p\sigma} = \phi_p \otimes \ket{\sigma}$, and we label the Fock space
basis by occupation numbers $\ket{n_0 n_1 \dots n_{2M-1}}$ with $n_p \in \{0,1\}$.

The fermionic operators $\hat{a}_p^\dagger, \hat{a}_p$ add and remove an electron from
spin orbital $p$ and obey

\begin{equation}
    \{\hat{a}_p, \hat{a}_q^\dagger\} = \delta_{pq}, \qquad
    \{\hat{a}_p, \hat{a}_q\} = \{\hat{a}_p^\dagger, \hat{a}_q^\dagger\} = 0 .
\end{equation}

The anticommutation is doing all the work: it *is* the Pauli principle and the
antisymmetry of the wavefunction, encoded in the algebra rather than in the states.
The Hamiltonian becomes

\begin{equation}
    \hat{H} = \sum_{pq,\sigma} h_{pq}\, \hat{a}^\dagger_{p\sigma}\hat{a}_{q\sigma}
    + \frac{1}{2}\sum_{pqrs}\sum_{\sigma\tau} (pq|rs)\,
      \hat{a}^\dagger_{p\sigma}\hat{a}^\dagger_{r\tau}\hat{a}_{s\tau}\hat{a}_{q\sigma}
    + E_{\rm nuc},
\end{equation}

where $\sigma,\tau \in \{\uparrow,\downarrow\}$ and the spatial indices run over the
$M$ orbitals. Mind the index order in the two-electron term: the integral is in
chemists' notation $(pq|rs)$, but the operators are *not* in the order $pqrs$. Getting
this wrong is the single most common bug in this whole pipeline, and it produces an
energy that looks plausible but is wrong by tens of mHa. We will check our result
against FCI precisely so that such a mistake cannot hide.

**Ordering convention.** We use *blocked* spin ordering: qubits $0 \dots M-1$ hold the
spin-up orbitals and qubits $M \dots 2M-1$ the spin-down ones. For $\mathrm{H}_2$ that
is four qubits, $(\phi_0\uparrow, \phi_1\uparrow, \phi_0\downarrow, \phi_1\downarrow)$,
and the Hartree--Fock state is $\ket{1010}$ in that labelling: one electron in the
bonding orbital of each spin.

---
# 3. The Jordan--Wigner transformation

Qubits and fermions are not the same thing. Qubit operators on *different* qubits
commute, whereas fermionic operators on different modes anticommute. Any mapping has
to repair that mismatch somewhere, and the Jordan--Wigner transformation does it by
attaching a string of $Z$ operators:

\begin{equation}
    \hat{a}_p = \left(\bigotimes_{q<p} \hat{Z}_q\right)
                \otimes \frac{\hat{X}_p + i\hat{Y}_p}{2}, \qquad
    \hat{a}^\dagger_p = \left(\bigotimes_{q<p} \hat{Z}_q\right)
                \otimes \frac{\hat{X}_p - i\hat{Y}_p}{2}.
\end{equation}

The local part $(\hat{X} \pm i\hat{Y})/2$ flips the occupation of mode $p$. The
$\hat{Z}$ string counts the parity of all the modes below $p$ and supplies the minus
signs that antisymmetry demands. It is also the reason Jordan--Wigner is expensive:
the string is $O(M)$ long, so a single fermionic excitation can become a Pauli operator
acting on the whole register. (Bravyi--Kitaev and parity mappings trade this off
differently, reducing the weight to $O(\log M)$.)

Occupation is read off from $Z$: $\hat{n}_p = \hat{a}^\dagger_p \hat{a}_p =
(\hat{I} - \hat{Z}_p)/2$, so $\ket{0}$ is empty and $\ket{1}$ is occupied.

Let us implement it. One wrinkle: Qiskit's Pauli label strings are **little-endian**,
so the rightmost character acts on qubit 0. We build the label with a list indexed by
qubit number and reverse it at the end.

In [ ]:
def jw_annihilation(p, n_qubits):
    '''The fermionic annihilation operator a_p as a SparsePauliOp, under Jordan-Wigner.'''
    def label(local_op):
        chars = ["I"] * n_qubits
        for q in range(p):          # the Z string on every mode below p
            chars[q] = "Z"
        chars[p] = local_op         # the local raising/lowering part
        return "".join(reversed(chars))   # reversed: Qiskit labels are little-endian

    return SparsePauliOp.from_list([(label("X"), 0.5), (label("Y"), 0.5j)])


def jw_creation(p, n_qubits):
    '''a_p^dagger is just the adjoint of a_p.'''
    return jw_annihilation(p, n_qubits).adjoint()


# What does a single operator look like?
print("a_0 on 4 qubits:", jw_annihilation(0, 4).to_list())
print("a_2 on 4 qubits:", jw_annihilation(2, 4).to_list())

### Exercise 1: check the algebra

Before trusting the mapping, verify that the operators we just built actually satisfy
the fermionic anticommutation relations. Fill in the cell below and confirm that

$$\{\hat{a}_p, \hat{a}_q^\dagger\} = \delta_{pq}\hat{I}, \qquad \{\hat{a}_p,\hat{a}_q\} = 0 .$$

Useful: `SparsePauliOp` supports `@` for the operator product, `+` for the sum, and
`.simplify()` to collect terms. A quick way to test whether an operator is zero is
`np.allclose(op.simplify().coeffs, 0)`.

In [ ]:
def anticommutator(A, B):
    return (A @ B + B @ A).simplify()


n_qubits = 4

# YOUR CODE HERE: loop over p, q and check the two relations.
#
# for p in range(n_qubits):
#     for q in range(n_qubits):
#         ...

In [ ]:
# Solution (run this if you want to check yours)
n_qubits = 4
a  = [jw_annihilation(p, n_qubits) for p in range(n_qubits)]
ad = [jw_creation(p, n_qubits)     for p in range(n_qubits)]

ok = True
for p in range(n_qubits):
    for q in range(n_qubits):
        expected = np.eye(2**n_qubits) if p == q else np.zeros((2**n_qubits,)*2)
        ok &= np.allclose(anticommutator(a[p], ad[q]).to_matrix(), expected)
        ok &= np.allclose(anticommutator(a[p], a[q]).to_matrix(), 0)

print("anticommutation relations satisfied:", ok)

## Assembling the Hamiltonian

Now we just transcribe the second-quantised Hamiltonian term by term, replacing every
$\hat{a}^\dagger_p$ and $\hat{a}_q$ by its Pauli representation.

In [ ]:
def build_qubit_hamiltonian(h1, eri, e_nuc):
    '''Map the molecular integrals to a qubit Hamiltonian via Jordan-Wigner.

    Blocked spin ordering: spatial orbital p, spin up  -> qubit p
                           spatial orbital p, spin down -> qubit p + M
    '''
    M = h1.shape[0]
    nq = 2 * M
    a  = [jw_annihilation(p, nq) for p in range(nq)]
    ad = [jw_creation(p, nq)     for p in range(nq)]

    terms = []

    # one-body:  sum_{pq, sigma} h_pq a^dag_{p sigma} a_{q sigma}
    for p in range(M):
        for q in range(M):
            if abs(h1[p, q]) < 1e-12:
                continue
            for sigma in (0, 1):
                terms.append(h1[p, q] * (ad[p + sigma*M] @ a[q + sigma*M]))

    # two-body:  1/2 sum_{pqrs, sigma tau} (pq|rs) a^dag_{p s} a^dag_{r t} a_{s t} a_{q s}
    for p in range(M):
        for q in range(M):
            for r in range(M):
                for s in range(M):
                    v = eri[p, q, r, s]
                    if abs(v) < 1e-12:
                        continue
                    for sigma in (0, 1):
                        for tau in (0, 1):
                            terms.append(0.5 * v * (
                                ad[p + sigma*M] @ ad[r + tau*M]
                                @ a[s + tau*M] @ a[q + sigma*M]))

    H = sum(terms[1:], terms[0])
    H = H + SparsePauliOp.from_list([("I" * nq, e_nuc)])   # nuclear repulsion
    return H.simplify(atol=1e-12)


H_4q = build_qubit_hamiltonian(h1, eri, e_nuc)

print(f"H acts on {H_4q.num_qubits} qubits and has {len(H_4q)} Pauli terms\n")
for label, coeff in sorted(H_4q.to_list(), key=lambda t: -abs(t[1])):
    print(f"  {label}   {coeff.real:+.6f}")

Two sanity checks you can do *by eye* on any real molecular Hamiltonian:

1. **All coefficients are real.** They must be, since $\hat{H}$ is Hermitian and the
   Pauli operators are Hermitian.
2. **Every term has an even number of $Y$s.** With real orbitals, the integrals are
   real, and $\hat{Y}$ is the only Pauli with imaginary matrix elements, so they have
   to pair up. A term like `YZ` (a single $Y$) cannot appear in an electronic
   Hamiltonian at all.

Hold on to check 2. We will come back to it.

## Does it reproduce the right physics?

The whole construction is only worth anything if the lowest eigenvalue of this
$16 \times 16$ matrix is the FCI energy we loaded earlier.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(H_4q.to_matrix())

print(f"lowest eigenvalue of H : {eigenvalues[0]:.8f} Ha")
print(f"FCI reference          : {e_fci:.8f} Ha")
print(f"difference             : {abs(eigenvalues[0] - e_fci):.2e} Ha")
assert abs(eigenvalues[0] - e_fci) < 1e-8, "mapping is wrong somewhere"
print("\nThe mapping is exact.")

### A word on what we just did

That agreement is not a small thing. We took integrals over Gaussian basis functions,
wrote a fermionic operator, replaced every mode with Pauli strings, and the ground
state of the resulting spin model is the ground state of a molecule to eight decimal
places. The map is exact, not an approximation. The only approximation in the whole
chain is the finite basis set.

Also note the size of the object: 15 Pauli terms for the smallest possible molecule.
The number of two-electron integrals grows as $M^4$, and so does the term count. For a
molecule anyone actually cares about you are looking at $10^6$--$10^9$ terms, every one
of which has to be measured. That is the real bottleneck of VQE for chemistry, and it
is a *measurement* problem rather than a circuit-depth problem.

---
# 4. Symmetries: getting from four qubits to two

Four qubits is a 16-dimensional Fock space, but we know the physical state we want has
exactly two electrons, one of each spin. Most of that space is irrelevant, and we can
exploit this.

The Hamiltonian conserves the number of spin-up electrons and the number of spin-down
electrons separately. In qubit language, that means it commutes with the parity
operators

\begin{equation}
    \hat{\Pi}_\uparrow = \hat{Z}_0 \hat{Z}_1, \qquad
    \hat{\Pi}_\downarrow = \hat{Z}_2 \hat{Z}_3,
\end{equation}

each of which measures $(-1)^{N_\sigma}$ for its spin sector. These generate a
$\mathbb{Z}_2 \times \mathbb{Z}_2$ symmetry group, and each independent $\mathbb{Z}_2$
lets us eliminate one qubit.

### Exercise 2: confirm the symmetry

Check that $[\hat{H}, \hat{\Pi}_\uparrow] = [\hat{H}, \hat{\Pi}_\downarrow] = 0$.
Remember the little-endian labels: $\hat{Z}_0\hat{Z}_1$ is the string `"IIZZ"` and
$\hat{Z}_2\hat{Z}_3$ is `"ZZII"`.

In [ ]:
def commutator(A, B):
    return (A @ B - B @ A).simplify(atol=1e-10)


# YOUR CODE HERE: build the two parity operators and check they commute with H_4q.

In [ ]:
# Solution
for name, string in [("Pi_up", "IIZZ"), ("Pi_down", "ZZII")]:
    comm = commutator(H_4q, SparsePauliOp(string))
    is_zero = np.allclose(comm.coeffs, 0)
    print(f"[H, {name}] = 0 :  {is_zero}")

## Tapering

The standard trick (Bravyi, Gambetta, Mezzacapo and Temme,
[arXiv:1701.08213](https://arxiv.org/abs/1701.08213)) is to rotate each symmetry
generator onto a single qubit, at which point that qubit decouples.

For a $Z$-type generator $\hat{\tau}$ and a chosen qubit $q$ on which $\hat{\tau}$ acts
non-trivially, define the Clifford

\begin{equation}
    \hat{U} = \frac{1}{\sqrt{2}}\left(\hat{X}_q + \hat{\tau}\right).
\end{equation}

$\hat{U}$ is both Hermitian and unitary, and it satisfies
$\hat{U}\hat{\tau}\hat{U}^\dagger = \hat{X}_q$. Since $[\hat{H},\hat{\tau}] = 0$, the
rotated Hamiltonian $\hat{U}\hat{H}\hat{U}^\dagger$ commutes with $\hat{X}_q$, which
means qubit $q$ only ever appears as $\hat{I}$ or $\hat{X}_q$. We can then replace
$\hat{X}_q$ by its eigenvalue $\pm 1$ and delete the qubit.

Which sign? That is the *sector*, and it is fixed by the physics: we want one spin-up
and one spin-down electron, so $N_\uparrow = N_\downarrow = 1$, both odd, so both
parities are $(-1)^1 = -1$.

In [ ]:
def taper(H, generators, qubits, sector):
    '''Remove one qubit per Z2 symmetry generator.

    generators : list of Z-type Pauli label strings commuting with H
    qubits     : the qubit each generator is rotated onto (and then removed)
    sector     : the +-1 eigenvalue of each generator for the state we want
    '''
    nq = H.num_qubits

    # Build the Clifford U = prod_i (X_{q_i} + tau_i)/sqrt(2)
    U = None
    for tau, q in zip(generators, qubits):
        x_q = "I" * (nq - 1 - q) + "X" + "I" * q
        U_i = SparsePauliOp.from_list([(tau, 1.0), (x_q, 1.0)]) * (1 / np.sqrt(2))
        U = U_i if U is None else U @ U_i

    H_rot = (U @ H @ U.adjoint()).simplify(atol=1e-10)

    # Now drop the decoupled qubits, replacing X_q -> sector value
    keep = [q for q in range(nq) if q not in qubits]
    reduced = {}
    for label, coeff in zip(H_rot.paulis.to_labels(), H_rot.coeffs):
        chars = list(reversed(label))            # chars[q] acts on qubit q
        sign = 1.0
        for q, value in zip(qubits, sector):
            if chars[q] == "X":
                sign *= value
            elif chars[q] != "I":
                raise ValueError(f"term {label} is not diagonal on the tapered qubits")
        new_label = "".join(reversed([chars[q] for q in keep]))
        reduced[new_label] = reduced.get(new_label, 0) + sign * coeff

    return SparsePauliOp.from_list(
        [(k, v) for k, v in reduced.items() if abs(v) > 1e-12]).simplify()


H_2q = taper(H_4q,
             generators=["IIZZ", "ZZII"],   # Pi_up, Pi_down
             qubits=[1, 3],                 # remove qubits 1 and 3
             sector=[-1, -1])               # N_up = N_down = 1, both odd

print("Two-qubit hydrogen Hamiltonian:\n")
for label, coeff in H_2q.to_list():
    print(f"  {label}   {coeff.real:+.6f}")

print(f"\nlowest eigenvalue : {np.linalg.eigvalsh(H_2q.to_matrix())[0]:.8f} Ha")
print(f"FCI reference     : {e_fci:.8f} Ha")

Two qubits, five terms, and still exact to machine precision. We have thrown away
three quarters of the Hilbert space and lost nothing, because the discarded part was
unphysical.

## Comparing with the Hamiltonian from notebook 2

Now put them side by side:

| | notebook 2 | derived here |
|---|---|---|
| | `YZ`  $+0.3980$ | `IZ`  $+0.397937$ |
| | `ZI`  $-0.3980$ | `ZI`  $+0.397937$ |
| | `ZZ`  $-0.0113$ | `ZZ`  $+0.011280$ |
| | `XX`  $+0.1810$ | `XX`  $+0.180931$ |
| | — | `II`  $-0.332404$ |

The magnitudes are the same to four decimal places, so the operator you were using in
notebook 2 is clearly a descendant of this one. But it is not the hydrogen molecule:

- The identity term is missing. That is a constant energy offset, which does not change
  the optimisation at all, but it does mean the number you minimised was not an energy.
- Some signs differ, which *does* change the spectrum.
- `YZ` has a single $\hat{Y}$. By the parity argument above, no real electronic
  Hamiltonian can contain such a term.

This is why the $-0.70$ you obtained in notebook 2 does not appear in any chemistry
textbook, while the $-1.137306$ we just computed is the STO-3G ground-state energy of
$\mathrm{H}_2$. Treat notebook 2 as a warm-up on a toy operator and this one as the
physics.

---
# 5. An ansatz that knows some chemistry

Notebook 1 distinguished *hardware-efficient* from *problem-inspired* ansätze. Here we
can see the difference concretely.

Look at the two-qubit Hamiltonian: it contains $\hat{I}$, $\hat{Z}_0$, $\hat{Z}_1$,
$\hat{Z}_0\hat{Z}_1$ and $\hat{X}_0\hat{X}_1$. The $\hat{X}\hat{X}$ term connects
$\ket{00}\leftrightarrow\ket{11}$ and $\ket{01}\leftrightarrow\ket{10}$, so the
Hamiltonian is block diagonal in parity, and the ground state lives entirely in the
even sector $\mathrm{span}\{\ket{00}, \ket{11}\}$.

Physically those two basis states are the Hartree--Fock determinant (both electrons in
the bonding orbital) and the doubly-excited determinant (both in the antibonding
orbital). The exact ground state is a superposition of just these two:

\begin{equation}
    \ket{\psi(\theta)} = \cos\tfrac{\theta}{2}\ket{00} + \sin\tfrac{\theta}{2}\ket{11},
\end{equation}

which is prepared by a single $R_y$ rotation followed by a CNOT. **One parameter, two
gates, and it can represent the exact ground state.** This is the two-qubit form of the
UCCD ansatz, and it is problem-inspired in the strongest possible sense: the chemistry
told us exactly which two-dimensional subspace to search.

In [ ]:
theta = Parameter("θ")

ansatz = QuantumCircuit(2)
ansatz.ry(theta, 0)
ansatz.cx(0, 1)

ansatz.draw("mpl", style="iqp")

Because there is only one parameter, we can plot the *entire* cost landscape. This is
a luxury you will never have again: for the 16-parameter `efficient_su2` of notebook 2
this surface lives in 16 dimensions.

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 400)
energies = [Statevector(ansatz.assign_parameters([t])).expectation_value(H_2q).real
            for t in thetas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thetas, energies, label=r"$\langle\psi(\theta)|H|\psi(\theta)\rangle$")
ax.axhline(e_fci, color="orange", ls="--", label=f"FCI = {e_fci:.5f} Ha")
ax.axhline(e_hf,  color="grey",   ls=":",  label=f"HF  = {e_hf:.5f} Ha")
ax.set_xlabel(r"$\theta$")
ax.set_ylabel("energy (Ha)")
ax.set_title(f"Cost landscape of the one-parameter ansatz, R = {R} $\\AA$")
ax.legend()
plt.show()

print(f"minimum on the grid : {min(energies):.6f} Ha")
print(f"FCI                 : {e_fci:.6f} Ha")

The curve touches the FCI line, to within the resolution of the $\theta$ grid,
confirming that the ansatz can represent the exact ground state. It is also smooth,
with a single well-separated minimum and no flat regions. Compare this to the
barren-plateau discussion in notebook 1: a landscape like this is the best case, and it
arises because we built the ansatz from physical knowledge instead of stacking generic
entangling layers.

Note also that $\theta = 0$ gives exactly the Hartree--Fock energy: $\ket{00}$ is the
HF determinant. So the VQE starting from $\theta = 0$ is literally starting from the
mean-field solution and improving on it.

---
# 6. Running the VQE

Same machinery as notebook 2: transpile to the backend's instruction set, wrap the
`Estimator` in a cost function, hand it to a classical optimiser.

In [ ]:
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
ansatz_isa = pm.run(ansatz)
H_2q_isa = H_2q.apply_layout(layout=ansatz_isa.layout)

estimator = Estimator(mode=backend)
estimator.options.default_shots = 10_000

history = {"iters": 0, "cost_history": []}


def cost_function(params, ansatz, hamiltonian, estimator):
    pub = (ansatz, [hamiltonian], [params])
    result = estimator.run(pubs=[pub]).result()
    energy = result[0].data.evs[0]

    history["iters"] += 1
    history["cost_history"].append(energy)
    return energy

In [ ]:
from functools import partial

wrapped = partial(cost_function, ansatz=ansatz_isa,
                  hamiltonian=H_2q_isa, estimator=estimator)

result = COBYLA(maxiter=100).minimize(fun=wrapped, x0=np.array([0.0]))

print(f"VQE energy   : {result.fun:.6f} Ha")
print(f"FCI energy   : {e_fci:.6f} Ha")
print(f"HF  energy   : {e_hf:.6f} Ha")
print(f"\nerror vs FCI : {1000*abs(result.fun - e_fci):.2f} mHa "
      f"(chemical accuracy is 1.6 mHa)")
print(f"optimal theta: {result.x[0]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(history["iters"]), history["cost_history"])
ax.axhline(e_fci, color="orange", ls="--", label="FCI")
ax.axhline(e_hf, color="grey", ls=":", label="HF")
ax.set_xlabel("cost function evaluations")
ax.set_ylabel("energy (Ha)")
ax.legend()
plt.show()

Look carefully at where the VQE energy lands relative to FCI. With 10 000 shots per
evaluation you will often find it sitting slightly *below* the exact answer, which
looks like a violation of the variational principle.

It is not. The variational bound applies to the true expectation value
$\bra{\psi}\hat{H}\ket{\psi}$, but what the optimiser sees is a noisy *estimate* of it,
with a standard error of order $10^{-2}$ Ha at this shot count. The optimiser will
happily walk towards a favourable fluctuation. This is worth internalising: with a
finite shot budget, "the VQE went below the exact energy" means your error bars are
larger than the effect you are looking for, not that you have found something.

Compare the standard error to chemical accuracy:

In [ ]:
pub_result = estimator.run([(ansatz_isa, [H_2q_isa], [result.x])]).result()
print(f"energy      : {pub_result[0].data.evs[0]:.6f} Ha")
print(f"std. error  : {pub_result[0].data.stds[0]:.6f} Ha  at 10,000 shots")
print(f"chemical accuracy target : 0.0016 Ha")

The estimator's error bar is several times larger than chemical accuracy. Since the
standard error falls as $1/\sqrt{N_{\rm shots}}$, reaching $1.6\ \mathrm{mHa}$ needs
roughly a hundred times more shots, for the smallest molecule in existence, on a
noiseless simulator. Scaling that thought to $10^6$ Hamiltonian terms is a good way to
understand why measurement reduction is such an active research area.

---
# 7. Exercise: the dissociation curve

Everything so far was at a single geometry. The interesting physics is in how the
energy varies as we pull the two atoms apart: the minimum gives the equilibrium bond
length, the curvature gives the vibrational frequency, and the large-$R$ limit gives
the dissociation energy.

Your task is to run a VQE at every bond length in the data file and plot the result
against the HF and FCI reference curves.

Three things to think about:

1. **You must rebuild the Hamiltonian at each geometry.** The integrals change with $R$,
   so `h1`, `eri` and `e_nuc` all change. The ansatz does not.
2. **Warm-starting.** The optimal $\theta$ varies smoothly with $R$, so starting each
   optimisation from the previous geometry's solution saves a lot of function
   evaluations. This is standard practice for potential energy surface scans.
3. **Use the exact `Statevector` expectation value rather than the shot-based estimator**
   for the scan, at least the first time. Otherwise shot noise (~10 mHa) will swamp the
   features you are trying to see (~20 mHa). Come back and add shots afterwards if you
   want to see how badly it degrades the curve.

In [ ]:
def h2_hamiltonian(index):
    '''Two-qubit tapered Hamiltonian for geometry `index` in the data file.'''
    return taper(
        build_qubit_hamiltonian(data["h1"][index], data["eri"][index], data["e_nuc"][index]),
        generators=["IIZZ", "ZZII"], qubits=[1, 3], sector=[-1, -1],
    )


# YOUR CODE HERE
#
# vqe_energies = []
# theta_start = 0.0
# for i, R_i in enumerate(bond_lengths):
#     H_i = h2_hamiltonian(i)
#     cost = lambda p: Statevector(ansatz.assign_parameters(p)).expectation_value(H_i).real
#     res = COBYLA(maxiter=100).minimize(fun=cost, x0=np.array([theta_start]))
#     ...

In [ ]:
# Solution
vqe_energies, thetas_opt = [], []
theta_start = 0.0

for i, R_i in enumerate(bond_lengths):
    H_i = h2_hamiltonian(i)

    def cost(params):
        return Statevector(ansatz.assign_parameters(params)).expectation_value(H_i).real

    res = COBYLA(maxiter=100).minimize(fun=cost, x0=np.array([theta_start]))
    vqe_energies.append(res.fun)
    thetas_opt.append(res.x[0])
    theta_start = res.x[0]          # warm start the next geometry

vqe_energies = np.array(vqe_energies)
print(f"largest deviation from FCI: {1000*np.max(np.abs(vqe_energies - data['e_fci'])):.4f} mHa")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(bond_lengths, data["e_hf"],  ":",  color="grey",   label="Hartree-Fock")
ax1.plot(bond_lengths, data["e_fci"], "-",  color="orange", label="FCI (exact)")
ax1.plot(bond_lengths, vqe_energies,  "o",  color="tab:blue", ms=4,
         mfc="none", label="VQE")
E_ATOMIC_STO3G = 2 * (-0.46658185)   # two isolated H atoms in STO-3G
ax1.axhline(E_ATOMIC_STO3G, color="k", lw=0.7, ls="--")
ax1.annotate("2 isolated H atoms, STO-3G", xy=(2.4, E_ATOMIC_STO3G),
             xytext=(1.9, E_ATOMIC_STO3G + 0.04), fontsize=8)
ax1.set_ylabel("energy (Ha)")
ax1.set_title(r"H$_2$ dissociation curve, STO-3G")
ax1.legend()

ax2.plot(bond_lengths, 1000*(data["e_hf"] - data["e_fci"]), ":", color="grey",
         label="HF error")
ax2.plot(bond_lengths, 1000*(vqe_energies - data["e_fci"]), "o", color="tab:blue",
         ms=4, mfc="none", label="VQE error")
ax2.axhline(1.6, color="green", ls="--", lw=1, label="chemical accuracy")
ax2.set_xlabel(r"bond length $R$ ($\AA$)")
ax2.set_ylabel("error vs FCI (mHa)")
ax2.set_yscale("symlog", linthresh=1e-2)
ax2.set_ylim(-1e-1, 1e3)
ax2.legend(loc="upper left")

plt.tight_layout()
plt.show()

## What the plot is telling you

**The VQE curve sits on top of FCI everywhere.** With an exact ansatz and no shot noise,
that is expected, but it is worth appreciating: a two-gate circuit reproduces the full
configuration interaction energy of a molecule across its entire dissociation curve.

**But FCI does not dissociate to $-1$ Ha.** Two isolated hydrogen atoms have an exact
energy of $-1$ Ha, and the curve flattens out around $-0.933$ Ha instead. That gap is
not an error in anything we did: it is the *basis set*. A single contraction of three
Gaussians cannot represent a $1s$ orbital exactly, so STO-3G gets the atom itself wrong
by 33 mHa. FCI is exact only within the basis you give it. Worth remembering when
comparing quantum algorithms against experiment: the basis set error usually dwarfs
everything the algorithm is doing.

**Hartree--Fock is fine near equilibrium and catastrophically wrong at large $R$.** At
$0.735\ \mathrm{\AA}$ it is off by about 20 mHa; by $3\ \mathrm{\AA}$ it is off by
hundreds. HF does not even dissociate to the right products: a restricted HF
calculation forces both electrons into the same spatial orbital, so at large $R$ it
describes an unphysical mixture that includes ionic $\mathrm{H}^+\mathrm{H}^-$
configurations.

This is **static correlation**, and it is exactly the regime where the ground state is
not well approximated by a single determinant. In our language, the optimal $\theta$
moves away from $0$: the doubly-excited determinant $\ket{11}$ acquires an amplitude
comparable to the reference $\ket{00}$. Have a look:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(bond_lengths, np.cos(np.array(thetas_opt)/2)**2, label=r"$|\langle 00|\psi\rangle|^2$ (HF determinant)")
ax.plot(bond_lengths, np.sin(np.array(thetas_opt)/2)**2, label=r"$|\langle 11|\psi\rangle|^2$ (double excitation)")
ax.set_xlabel(r"bond length $R$ ($\AA$)")
ax.set_ylabel("weight in the ground state")
ax.axhline(0.5, color="k", lw=0.5, ls="--")
ax.legend()
ax.set_title("Breakdown of the single-determinant picture")
plt.show()

Near equilibrium the state is roughly 98% Hartree--Fock, so mean-field theory is a good
starting point and perturbative corrections work well. As the bond stretches the two
determinants become degenerate and the weights approach 50/50, at which point *no*
single-determinant method can work and the perturbative expansion diverges.

Multireference problems of this kind, in transition metal complexes and bond-breaking
processes, are the standard argument for why quantum computers might be useful for
chemistry: they are the cases where the classical approximations that we know how to
scale are the ones that break down.

---
# Where to go next

Some directions, roughly in order of effort:

**Add shot noise to the scan.** Swap `Statevector` back for the `Estimator` with 10 000
shots and redo the dissociation curve. The curve becomes visibly ragged and the
correlation energy is no longer resolvable. Then work out how many shots you would need
per point to get chemical accuracy, and how long that would take at a realistic circuit
repetition rate.

**Skip the tapering and run all four qubits.** Use `efficient_su2(4)` from notebook 2 on
`H_4q` instead. It has 32 parameters, does not conserve particle number, and can
converge to a state in the wrong sector entirely. Compare the number of iterations
needed against the one-parameter ansatz. This is the clearest demonstration available
of what problem-inspired ansätze buy you.

**Change the mapping.** Implement the parity mapping, where the qubit stores
$\sum_{q\le p} n_q \bmod 2$ instead of $n_p$, and check that you get the same spectrum
with a different set of Pauli strings. Then compare the average Pauli weight.

**Change the molecule.** `data/generate_integrals.py` will produce integrals for
anything PySCF can handle. LiH in STO-3G is the usual next step: 6 spatial orbitals,
12 qubits before tapering, and freezing the lithium core is required to make it
tractable.

**Excited states.** The VQE finds the ground state. Getting excited states needs a
different objective, for example VQD, which adds an overlap penalty
$\beta|\langle\psi_0|\psi(\boldsymbol{\theta})\rangle|^2$ to push the optimisation away
from states you have already found. For $\mathrm{H}_2$ the first excited state in the
even-parity sector is just the orthogonal combination, so you can check the answer
analytically.